# EDA Completo v4 - Dataset Sintético
## Análisis de Calidad con Réplica de 4 Fuentes Auditoria3

Análisis exploratorio del dataset sintético generado replicando las 4 fuentes independientes:
- **Vehiculo**: Fleet data (27 columns) from flota/*.xlsx
- **Dispositivo**: Telemetry devices (21 columns) from telemetria/*.xlsx
- **Reporte Consumo**: Fuel consumption reports (31 columns) from consumo/ReporteConsumo/*.xlsx
- **Solicitud Combustible**: Fuel requests (24 columns) from consumo/Solicitudes/*.xlsx

Validaciones:
- Schema completeness and column alignment
- Nullability patterns matching real data
- Defect injection verification
- Cross-source key coverage (matricula, dominio matching)
- Data quality statistics

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

drive.mount('/content/drive')
print("✓ Google Drive mounted")

In [ ]:
# Load dataset
dataset_dir = Path('/content/drive/MyDrive/Integrador/datasets/complete_v4')

# Load all CSVs
tables = {}
for csv_file in sorted(dataset_dir.glob('*.csv')):
    if 'ejecucion' not in csv_file.name:  # Skip execution log for now
        table_name = csv_file.stem
        tables[table_name] = pd.read_csv(csv_file)
        print(f"✓ {table_name:30s} {len(tables[table_name]):6d} rows × {len(tables[table_name].columns):2d} cols")

# Load manifest
manifest_path = dataset_dir / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\n✓ Manifest loaded (seed: {manifest['seed']})")

print(f"\n📊 Total records: {sum(len(df) for df in tables.values()):,}")

## 1. SCHEMA DOCUMENTATION

In [ ]:
def analyze_table_schema(df, table_name):
    """Detailed schema analysis for a single table"""
    print(f"\n{'='*100}")
    print(f"TABLE: {table_name.upper()}")
    print(f"{'='*100}")
    print(f"Dimensions: {len(df):6d} rows × {len(df.columns):2d} columns")
    print(f"\nCOLUMN DETAILS:")
    print(f"{'#':>3} {'Column Name':30} {'Type':15} {'Non-Null':>10} {'Null %':>8} {'Unique':>10}")
    print("-" * 100)
    
    for i, (col, dtype) in enumerate(df.dtypes.items(), 1):
        non_null = df[col].notna().sum()
        null_pct = 100 * df[col].isna().sum() / len(df)
        unique = df[col].nunique()
        marker = '⚠ NULL' if null_pct > 50 else '✓ DATA'
        print(f"{i:3d} {col:30s} {str(dtype):15s} {non_null:10d} {null_pct:7.1f}% {unique:10d} {marker}")

# Analyze all tables
for table_name, df in tables.items():
    analyze_table_schema(df, table_name)

## 2. NULLABILITY ANALYSIS

In [ ]:
# Create nullability summary
print("\n📊 NULLABILITY SUMMARY")
print("\n")

for table_name, df in tables.items():
    print(f"\n{table_name.upper()}:")
    null_summary = pd.DataFrame({
        'Total': len(df),
        'Non-Null': df.notna().sum(),
        'Null': df.isna().sum(),
        'Null %': (100 * df.isna().sum() / len(df)).round(1)
    })
    
    # Sort by null percentage descending
    null_summary = null_summary.sort_values('Null %', ascending=False)
    
    # Show columns with nulls
    cols_with_nulls = null_summary[null_summary['Null'] > 0]
    if len(cols_with_nulls) > 0:
        print(f"Columns with nulls ({len(cols_with_nulls)}):")
        for col in cols_with_nulls.index:
            print(f"  {col:30s} {cols_with_nulls.loc[col, 'Null']:6.0f} ({cols_with_nulls.loc[col, 'Null %']:5.1f}%)")
    else:
        print(f"✓ No null values")
    
    # Show completely empty columns
    empty_cols = null_summary[null_summary['Non-Null'] == 0]
    if len(empty_cols) > 0:
        print(f"\n⚠ COMPLETELY EMPTY COLUMNS ({len(empty_cols)}):")
        for col in empty_cols.index:
            print(f"  {col:30s} 100% empty")

In [ ]:
# Visualize nullability patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Nullability Patterns by Table', fontsize=14, fontweight='bold')

for idx, (table_name, df) in enumerate(tables.items()):
    ax = axes[idx // 2, idx % 2]
    
    # Calculate null percentages
    null_pct = (100 * df.isna().sum() / len(df)).sort_values(ascending=False)
    
    # Filter to columns with some nulls
    cols_with_nulls = null_pct[null_pct > 0]
    
    if len(cols_with_nulls) > 0:
        cols_with_nulls.plot(kind='barh', ax=ax, color='#FF6B6B')
        ax.set_title(f'{table_name.upper()} - Null %', fontweight='bold')
        ax.set_xlabel('Null Percentage (%)')
        ax.set_xlim(0, 105)
    else:
        ax.text(0.5, 0.5, 'No null values', ha='center', va='center', fontsize=12)
        ax.set_title(f'{table_name.upper()} - No Nulls', fontweight='bold')
        ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 3. VEHICULO TABLE ANALYSIS

In [ ]:
if 'vehiculo' in tables:
    vehiculos = tables['vehiculo']
    print("🔍 VEHICULO ANALYSIS")
    print(f"\nTotal vehicles: {len(vehiculos)}")
    
    # Domain analysis
    print(f"\n📋 DOMAIN VALIDATION:")
    dominios_unicos = vehiculos['Dominio'].nunique()
    dominios_duplicados = len(vehiculos) - dominios_unicos
    
    print(f"  Unique domains: {dominios_unicos}")
    print(f"  Duplicate domains: {dominios_duplicados}")
    print(f"  Duplication rate: {100*dominios_duplicados/len(vehiculos):.2f}%")
    
    # Top duplicates
    dup_count = vehiculos['Dominio'].value_counts()
    dup_count = dup_count[dup_count > 1]
    
    if len(dup_count) > 0:
        print(f"\n  Top 5 domains with duplicates:")
        for dom, cnt in dup_count.head(5).items():
            print(f"    {dom:20s} {cnt:3d} vehicles")
    
    # Format validation
    import re
    valid_pattern = r'^[A-Z]{2}\d{3}[A-Z]{2}$'
    invalid = vehiculos[~vehiculos['Dominio'].str.match(valid_pattern, na=False)]
    print(f"\n  Format validation (XX###XX):")
    print(f"    Valid format: {len(vehiculos) - len(invalid)} ({100*(len(vehiculos)-len(invalid))/len(vehiculos):.1f}%)")
    print(f"    Invalid format: {len(invalid)} ({100*len(invalid)/len(vehiculos):.1f}%)")
    
    # Year analysis
    print(f"\n  Year distribution:")
    año_counts = vehiculos['Año'].value_counts().sort_index(ascending=False)
    for año, cnt in año_counts.head(5).items():
        print(f"    {año:4s} {cnt:3d} vehicles")


## 4. DISPOSITIVO TABLE ANALYSIS

In [ ]:
if 'dispositivo' in tables:
    dispositivos = tables['dispositivo']
    print("📡 DISPOSITIVO ANALYSIS")
    print(f"\nTotal devices: {len(dispositivos)}")
    
    # Device by group
    print(f"\n📊 Devices by group:")
    grupo_counts = dispositivos['Grupo'].value_counts()
    for grupo, count in grupo_counts.items():
        print(f"  {grupo:20s} {count:3d}")
    
    # Coverage
    if 'vehiculo' in tables:
        vehiculos = tables['vehiculo']
        veh_with_dev = 0
        for placa in dispositivos['Placa'].unique():
            if placa in vehiculos['Dominio'].values:
                veh_with_dev += 1
        
        veh_without_dev = len(vehiculos) - veh_with_dev
        
        print(f"\n🔗 Fleet coverage:")
        print(f"  Vehicles with device: {veh_with_dev} ({100*veh_with_dev/len(vehiculos):.1f}%)")
        print(f"  Vehicles without device: {veh_without_dev} ({100*veh_without_dev/len(vehiculos):.1f}%)")
        print(f"  Avg devices per vehicle: {len(dispositivos)/veh_with_dev:.2f}")


## 5. REPORTE CONSUMO ANALYSIS

In [ ]:
if 'reporte_consumo' in tables:
    consumo = tables['reporte_consumo']
    print("⛽ REPORTE CONSUMO ANALYSIS")
    print(f"\nTotal fuel reports: {len(consumo)}")
    
    # Date range
    print(f"\n📅 Date analysis:")
    try:
        consumo['FECHA'] = pd.to_datetime(consumo['FECHA'], errors='coerce')
        print(f"  Date range: {consumo['FECHA'].min()} to {consumo['FECHA'].max()}")
        print(f"  Invalid dates: {consumo['FECHA'].isna().sum()}")
    except:
        print(f"  Could not parse dates")
    
    # Liters analysis
    print(f"\n📊 Fuel volume statistics:")
    try:
        liters = pd.to_numeric(consumo['LITROS UNIDADES'], errors='coerce')
        print(f"  Mean: {liters.mean():.2f} L")
        print(f"  Median: {liters.median():.2f} L")
        print(f"  Std Dev: {liters.std():.2f} L")
        print(f"  Range: [{liters.min():.2f}, {liters.max():.2f}] L")
        print(f"  Invalid: {liters.isna().sum()}")
    except:
        print(f"  Could not parse liters")
    
    # Stations
    print(f"\n🏪 Station distribution:")
    if 'ESTACION' in consumo.columns:
        estacion_counts = consumo['ESTACION'].value_counts()
        for estacion, count in estacion_counts.head(5).items():
            print(f"  {str(estacion):20s} {count:3d}")


## 6. SOLICITUD COMBUSTIBLE ANALYSIS

In [ ]:
if 'solicitud_combustible' in tables:
    solicitudes = tables['solicitud_combustible']
    print("📝 SOLICITUD COMBUSTIBLE ANALYSIS")
    print(f"\nTotal fuel requests: {len(solicitudes)}")
    
    # Matricula vs actual liters
    print(f"\n⚠️ COMPLIANCE CHECK:")
    try:
        solicitudes['LitrosAutorizados'] = pd.to_numeric(solicitudes['LitrosAutorizados'], errors='coerce')
        solicitudes['LitrosCargados'] = pd.to_numeric(solicitudes['LitrosCargados'], errors='coerce')
        
        # Over-loads (charged more than authorized)
        overloads = solicitudes[solicitudes['LitrosCargados'] > solicitudes['LitrosAutorizados']]
        print(f"  Requests with over-charging: {len(overloads)} ({100*len(overloads)/len(solicitudes):.1f}%)")
        
        # Under-loads
        underloads = solicitudes[solicitudes['LitrosCargados'] < solicitudes['LitrosAutorizados']]
        print(f"  Requests with under-charging: {len(underloads)} ({100*len(underloads)/len(solicitudes):.1f}%)")
        
        # Exact match
        exact = solicitudes[solicitudes['LitrosCargados'] == solicitudes['LitrosAutorizados']]
        print(f"  Requests with exact match: {len(exact)} ({100*len(exact)/len(solicitudes):.1f}%)")
    except Exception as e:
        print(f"  Could not analyze liters: {e}")
    
    # Date range
    print(f"\n📅 Date range:")
    try:
        solicitudes['FechaRendicion'] = pd.to_datetime(solicitudes['FechaRendicion'], errors='coerce')
        print(f"  From: {solicitudes['FechaRendicion'].min()}")
        print(f"  To: {solicitudes['FechaRendicion'].max()}")
        print(f"  Invalid dates: {solicitudes['FechaRendicion'].isna().sum()}")
    except:
        print(f"  Could not parse dates")


## 7. CROSS-TABLE ANALYSIS

In [ ]:
print("🔗 CROSS-TABLE KEY MATCHING")
print("\n")

# Vehiculo vs Dispositivo
if 'vehiculo' in tables and 'dispositivo' in tables:
    vehiculos = tables['vehiculo']
    dispositivos = tables['dispositivo']
    
    vehiculos_set = set(vehiculos['Dominio'].unique())
    dispositivos_set = set(dispositivos['Placa'].unique())
    
    matching = vehiculos_set & dispositivos_set
    missing = dispositivos_set - vehiculos_set
    
    print(f"VEHICULO ↔ DISPOSITIVO:")
    print(f"  Vehicles (unique domains): {len(vehiculos_set)}")
    print(f"  Devices (unique plaques): {len(dispositivos_set)}")
  print(f"  Matched: {len(matching)} ({100*len(matching)/len(dispositivos_set):.1f}%)")
    print(f"  Device plaques not in fleet: {len(missing)} ({100*len(missing)/len(dispositivos_set):.1f}%)")

# Solicitud vs Vehiculo
if 'solicitud_combustible' in tables and 'vehiculo' in tables:
    solicitudes = tables['solicitud_combustible']
    vehiculos = tables['vehiculo']
    
    solicitudes_set = set(solicitudes['Dominio'].dropna().unique())
    vehiculos_set = set(vehiculos['Dominio'].unique())
    
    matching = solicitudes_set & vehiculos_set
    missing = solicitudes_set - vehiculos_set
    
    print(f"\nSOLICITUD ↔ VEHICULO:")
    print(f"  Solicitudes (unique domains): {len(solicitudes_set)}")
    print(f"  Vehicles (unique domains): {len(vehiculos_set)}")
    print(f"  Matched: {len(matching)} ({100*len(matching)/len(solicitudes_set):.1f}%)")
    print(f"  Solicitud domains not in fleet: {len(missing)} ({100*len(missing)/len(solicitudes_set):.1f}%)")


## 8. DATA QUALITY SUMMARY

In [ ]:
print("\n" + "="*100)
print("📊 EXECUTIVE SUMMARY - SYNTHETIC DATASET QUALITY")
print("="*100)

print(f"\n📈 VOLUME:")
for table_name, df in tables.items():
    print(f"  {table_name:30s} {len(df):8d} records × {len(df.columns):2d} fields")

total_records = sum(len(df) for df in tables.values())
print(f"  {'TOTAL':30s} {total_records:8d} records")

print(f"\n✅ COMPLETENESS:")
for table_name, df in tables.items():
    total_cells = len(df) * len(df.columns)
    non_null_cells = df.notna().sum().sum()
    completeness = 100 * non_null_cells / total_cells
    print(f"  {table_name:30s} {completeness:6.1f}% complete")

print(f"\n🎯 KEY COVERAGE:")
if 'vehiculo' in tables and 'dispositivo' in tables:
    vehiculos = tables['vehiculo']
    dispositivos = tables['dispositivo']
    veh_set = set(vehiculos['Dominio'].unique())
    dev_set = set(dispositivos['Placa'].unique())
    coverage = len(veh_set & dev_set) / len(dev_set) * 100
    print(f"  Vehiculo ↔ Dispositivo coverage: {coverage:.1f}%")

if 'solicitud_combustible' in tables and 'vehiculo' in tables:
    solicitudes = tables['solicitud_combustible']
    vehiculos = tables['vehiculo']
    sol_set = set(solicitudes['Dominio'].dropna().unique())
    veh_set = set(vehiculos['Dominio'].unique())
    coverage = len(sol_set & veh_set) / len(sol_set) * 100 if len(sol_set) > 0 else 0
    print(f"  Solicitud ↔ Vehiculo coverage: {coverage:.1f}%")

if 'manifest' in locals():
    print(f"\n📝 GENERATION INFO:")
    print(f"  Generated: {manifest.get('generated_at', 'N/A')}")
    print(f"  Seed: {manifest.get('seed')}")
    print(f"  Config: {manifest.get('config', {})}")

print("\n" + "="*100)

## 9. VISUALIZATIONS

In [ ]:
# Table size comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

# Table sizes
table_sizes = {name: len(df) for name, df in tables.items()}
axes[0].bar(table_sizes.keys(), table_sizes.values(), color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
axes[0].set_title('Records per Table', fontweight='bold')
axes[0].set_ylabel('Number of Records')
axes[0].tick_params(axis='x', rotation=45)

# Column counts
col_counts = {name: len(df.columns) for name, df in tables.items()}
axes[1].bar(col_counts.keys(), col_counts.values(), color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
axes[1].set_title('Columns per Table', fontweight='bold')
axes[1].set_ylabel('Number of Columns')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Completeness heatmap
completeness_data = []
for table_name, df in tables.items():
    completeness = []
    for col in df.columns:
        completeness.append(100 * df[col].notna().sum() / len(df))
    completeness_data.append(completeness)

fig, ax = plt.subplots(figsize=(12, 8))

# Create data for heatmap (transpose so tables are rows, columns are x-axis)
max_cols = max(len(df.columns) for df in tables.values())
heatmap_data = []
labels = list(tables.keys())

for table_name, df in tables.items():
    row = []
    for col in df.columns:
        completeness = 100 * df[col].notna().sum() / len(df)
        row.append(completeness)
    heatmap_data.append(row)

# Create simplified heatmap (showing first 15 columns to avoid overcrowding)
heatmap_data_trimmed = [row[:15] for row in heatmap_data]
col_labels_trimmed = [col[:15] for col in list(tables.values())[0].columns[:15]]

sns.heatmap(heatmap_data_trimmed, annot=False, cmap='RdYlGn', vmin=0, vmax=100,
            yticklabels=labels, xticklabels=col_labels_trimmed, cbar_kws={'label': 'Completeness %'})
plt.title('Data Completeness by Table (first 15 columns)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()